# Objectives 

- Familiarization with smart contarcts 
- Step through OPCODE level execution of a transaction
- Familiarize and set up web3 library and wallet APIs 
- Interacting with an ERC20 token



# Walkthrough smart contract

We have deployed smart contracts for the ERC20 tokens on the Sepolia testnet. The contracts can be found on etherscan here: [LUSD](https://sepolia.etherscan.io/address/0xa2039612e374ad2365361DF0FF34399A4f0A1e58), [TIGER](https://sepolia.etherscan.io/address/0x706d4199f1333258F6a3386Db87792Ff6AcB04Ae). We will pretend that the Lab USD token has a value of 1 USD, and the TIGER token value is decided by the market (i.e. participants such as you buying/selling these tokens). 

Let us now go through the contract and describe what each ```function``` does.

### What does `balanceOf` and `totalSupply` do?

`balanceOf(address)` returns the number of tokens held by a specific wallet. `totalSupply()` returns the total number of tokens that exist across the entire contract, i.e., the sum of all holders' balances, minus any burned tokens.

In an ERC-20 contract, these are the core read-only functions used to check wallet holdings and the overall token supply.


### What does `transfer` and `transferFrom` do ? Which of them needs you to run `approve` before it executes correctly ?

`transfer(to, amount)` moves tokens from the caller's wallet to another address. It is a direct transfer from the sender to the recipient.

`transferFrom(from, to, amount)` moves tokens from one address to another on behalf of the owner. This is how a contract or exchange can spend tokens that belong to another wallet, but only after the owner has granted permission using `approve`.

`approve(spender, amount)` must be called before `transferFrom` succeeds. `transfer` does not require prior approval because the tokens are being sent directly by the sender.

The workflow is usually:

1. `approve(spender, amount)` from the token owner
2. `transferFrom(owner, recipient, amount)` from the spender

This is how ERC-20 standard wallets and DeFi contracts manage allowance-based token movement.


### What events can be emitted by the contract ? [What is their use](https://consensys.net/blog/developers/guide-to-events-and-logs-in-ethereum-smart-contracts/) ?

A standard ERC-20 token contract emits at least:

- `Transfer(address indexed from, address indexed to, uint256 value)`
- `Approval(address indexed owner, address indexed spender, uint256 value)`

These events are used for off-chain tracking and indexing. Applications, wallets, and front ends can listen for them to know when balances changed or when an address authorized a spender. They are stored in transaction logs and can be queried without loading the full blockchain state.


### How does the contract respond to airdrop requests ?

The airdrop function typically checks whether the caller is eligible, whether the request is within the allowed time window, and whether the token faucet still has enough balance to distribute. If the conditions are satisfied, the contract transfers a fixed token amount to the requesting wallet and records the request timestamp or registration status.

In the provided faucet design, each wallet can request tokens only once within a time period, and the contract enforces that by checking the last request timestamp and the faucet's remaining token balance before minting/transferring the tokens.


# Requesting tokens

To request the tokens we shall be using in the future for some in-class labs, we are going to use the ```web3``` library in python. The installation instructions can be found [here](https://web3py.readthedocs.io/en/v5/quickstart.html). We will view our transactions on the [Sepolia testnet etherscan](https://sepolia.etherscan.io/). 

We now send a transaction that uses this API to request some of the Lab ERC20 tokens from a [faucet contract](https://sepolia.etherscan.io/address/0x681878325277B9cC3cFdD3922321AcedD94D97b9). The faucet allows each registered wallet to redeem a fixed number of tokens per day. The code used to do that has to be written below.

### Setup Web3 API
First, we connect your wallet to a Web3 API - this is a service that connects you to the Ethereum blockchain via a simple URL. 

To do that, first create a free account on [Infura](https://app.infura.io/register)(using any email ID).

Once you log in, you would see your Infura API key under the project name you had given. Use that key in the following code.

In [1]:
# Import web3
from web3 import Web3

# Replace with your Infura or Alchemy Sepolia endpoint
RPC_URL = "https://sepolia.infura.io/v3/01efe8e6393f4d2bb5340bc78aa92460"
w3 = Web3(Web3.HTTPProvider(RPC_URL))

print("Connected:", w3.is_connected())
print("Latest block:", w3.eth.block_number)


Connected: True
Latest block: 11759507


### Setup your ETH account details

In [2]:
# Wallet details

public_address = Web3.to_checksum_address("0xC4ecbe357F8F75E867fdDE8fDb078be2a7e48203") # see metamask for this
private_key = 0xa1fecfad4e083e51246ca18a265ee911a2b6424a45bee4e654ddac325767eb1e # see metamask for this

### Set up a contract instance
Now, we send a request transaction to the faucet, signed by your private key.

In [3]:
# Define the TigerFaucet contract to connect to

contract_addr = Web3.to_checksum_address("0x681878325277B9cC3cFdD3922321AcedD94D97b9")
abi_contract = '[{"inputs":[],"stateMutability":"nonpayable","type":"constructor"},{"inputs":[{"internalType":"address[]","name":"user_list","type":"address[]"},{"internalType":"address","name":"token_address","type":"address"}],"name":"deRegisterUsers","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"user_address","type":"address"},{"internalType":"address","name":"token_address","type":"address"}],"name":"getLatestRequestTimestamp","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"user_address","type":"address"},{"internalType":"address","name":"token_address","type":"address"}],"name":"getRegistrationStatus","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"manager","outputs":[{"internalType":"address","name":"","type":"address"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address[]","name":"user_list","type":"address[]"},{"internalType":"address","name":"token_address","type":"address"}],"name":"registerUsers","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"contract IERC20","name":"token","type":"address"}],"name":"requestTokens","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"contract IERC20","name":"token","type":"address"}],"name":"retrieveFaucetAmount","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"contract IERC20","name":"token","type":"address"}],"name":"retrieveFaucetBalance","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"token_address","type":"address"},{"internalType":"uint256","name":"value","type":"uint256"}],"name":"set_faucet_amount","outputs":[],"stateMutability":"nonpayable","type":"function"}]' # see the "Code" section of the faucet contract on etherscan
contract = w3.eth.contract(contract_addr, abi=abi_contract)

# Define the address of the ERC20 token you want

token_addr = Web3.to_checksum_address("0x0F47E03a0383396635846f8495003fc478b71bcA")


### Create transaction data

In [5]:
### Create transaction data
txn = contract.functions.requestTokens(token_addr).build_transaction(
    {
        'nonce':w3.eth.get_transaction_count(public_address),
        'from':public_address,
    }
)

ContractLogicError: ('execution reverted: Only registered users can use this airdrop', '0x08c379a00000000000000000000000000000000000000000000000000000000000000020000000000000000000000000000000000000000000000000000000000000002a4f6e6c7920726567697374657265642075736572732063616e2075736520746869732061697264726f7000000000000000000000000000000000000000000000')

### Sign and post transaction
You should see the hash of your transaction printed.

In [12]:
# Post transaction

signed_txn = w3.eth.account.sign_transaction(txn, private_key) # Sign the txn with your private key 
txn_hash = w3.eth.send_raw_transaction(signed_txn.raw_transaction) # Send it onto the chain


In [13]:
print("Hash of swap transaction : ",Web3.to_hex(txn_hash)) # Search the txn hash on etherscan to see if it got confirmed

Hash of swap transaction :  0x57621e6b3531db03fe824ea9c145c01b7b7901accb8dae2fe5768e8234685eb8


## Request tokens through Etherscan

Verified contracts on Etherscan allow you to interact with functions on the contract.
Open the [Tiger Faucet](https://sepolia.etherscan.io/address/0x10831292abfc6dd365d51613afeaa80ded653645) contract by providing its address. You can now access the contract's read and write functions.

Go to the `Write` tab, connect your MetaMask wallet, and call `requestTokens` with the token contract address you want (for example, the TIGER token address). The faucet uses `msg.sender` to determine the requesting wallet, so you do not pass your wallet address as a function argument in this function.

After the transaction confirms, you can verify the token transfer on Etherscan and add the token to MetaMask.


# Add to MetaMask

- The token balances can now be tracked via MetaMask using the contract addresses of each token. 
- Use the "Import Tokens" link at the bottom of your MetaMask. 
- Enter the smart contract address for an ERC20 tokens we would be using in the class.
- Do this for each of the tokens - TIGER and LUSD

## Lab completion checklist

- Understand ERC-20 behavior: `balanceOf`, `totalSupply`, `transfer`, `transferFrom`, `approve`, and emitted events.
- Connect to Sepolia with Web3 and confirm a valid RPC connection.
- Use the faucet contract to request tokens for your registered wallet.
- Import the token into MetaMask using the token contract address.
- Confirm the transfer on Sepolia Etherscan.

This completes the Week 2 lab workflow for interacting with ERC-20 tokens on a public testnet.
